In [1]:

# LLOYDS BANK PROJECT
# Model Experiments

# This notebook evaluates additional machine learning models
# beyond the baseline experiments.

# Models:
# 1. Extra Trees
# 2. AdaBoost
# 3. HistGradientBoosting


# Team Member: Soham Pote


In [2]:

# IMPORT LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

# Machine Learning Models
from sklearn.ensemble import (
    ExtraTreesClassifier,
    AdaBoostClassifier,
    HistGradientBoostingClassifier
)

# Data splitting
from sklearn.model_selection import train_test_split

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [3]:

# LOAD DATASET

print("Loading dataset...")

df = pd.read_csv(
    "output/feature_engineered_dataset.csv",
    low_memory=False
)

print("✅ Dataset Loaded Successfully!")

print("\nRows :", df.shape[0])
print("Columns :", df.shape[1])

print("\nFirst five rows:")
display(df.head())

Loading dataset...
✅ Dataset Loaded Successfully!

Rows : 3145434
Columns : 45

First five rows:


,Mortgages.NumMortCharges,Mortgages.NumMortOutstanding,Mortgages.NumMortPartSatisfied,Mortgages.NumMortSatisfied,sic_code_1,bcb_sector,company_age_years,accounts_overdue_days,accounts_ever_late,conf_stmt_overdue_days,...,sector_avg_age,sector_mortgage_rate,sector_overdue_rate,sector_dormant_rate,age_vs_sector_avg,sic_division,active_with_mortgages,small_and_growing,overdue_and_active,full_accounts_with_charges
0,0,0,0,0,62020,Technology_Legal_Professional,1.530459,0.0,0,0.0,...,9.038258,0.048175,0.058985,0.117957,-7.507800,62,0,0,0,0
1,0,0,0,0,59112,Technology_Legal_Professional,7.430527,0.0,0,0.0,...,9.038258,0.048175,0.058985,0.117957,-1.607731,59,0,0,0,0
2,0,0,0,0,86210,Healthcare,0.520192,0.0,0,0.0,...,7.469492,0.090836,0.053380,0.078756,-6.949301,86,0,0,0,0
3,0,0,0,0,70229,Technology_Legal_Professional,2.800821,0.0,0,0.0,...,9.038258,0.048175,0.058985,0.117957,-6.237437,70,0,0,0,0
4,0,0,0,0,58290,Technology_Legal_Professional,4.867898,0.0,0,0.0,...,9.038258,0.048175,0.058985,0.117957,-4.170360,58,0,0,0,0


In [4]:

# PREPARE DATA FOR GROWTH OPPORTUNITY MODEL


target_column = "label_growth_opportunity"

# Remove all target/label columns from the model inputs
label_columns = [
    "label_growth_opportunity",
    "label_risk_signal",
    "label_lending_need_proxy"
]

X = df.drop(columns=label_columns)
y = df[target_column]

# Keep only numeric columns
X = X.select_dtypes(include=["number", "bool"])

# Replace missing values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

print("✅ Growth modelling data prepared")
print("Features:", X.shape[1])
print("Rows:", X.shape[0])
print("\nTarget distribution:")
print(y.value_counts())

✅ Growth modelling data prepared
Features: 41
Rows: 3145434

Target distribution:
label_growth_opportunity
1    1782383
0    1363051
Name: count, dtype: int64


In [5]:

# TRAIN / TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("✅ Train/Test split completed!")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

✅ Train/Test split completed!
Training samples: 2516347
Testing samples: 629087


In [6]:

# EXTRA TREES MODEL

extra_model = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

print("Training Extra Trees model...")

extra_model.fit(X_train, y_train)

y_pred = extra_model.predict(X_test)
y_prob = extra_model.predict_proba(X_test)[:, 1]

print("\n✅ Extra Trees Results")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_prob))

Training Extra Trees model...

✅ Extra Trees Results
Accuracy : 0.999747252764721
Precision: 0.9996970835881009
Recall   : 0.9998569332663818
F1 Score : 0.9997770020378367
ROC AUC  : 0.999801703960339


In [7]:

# USE LEAKAGE-SAFE FEATURES FOR GROWTH MODEL

GROWTH_FEATURES = [
    "has_mortgage_history",
    "mortgage_completion_rate",
    "mortgage_active_ratio",
    "mortgage_per_year",
    "company_size_score",
    "age_vs_sector_avg",
    "is_severely_overdue",
    "overdue_intensity",
    "sector_avg_age",
    "sector_mortgage_rate",
    "sector_overdue_rate",
    "sector_dormant_rate",
    "sic_division",
    "active_with_mortgages",
    "small_and_growing",
    "overdue_and_active",
    "full_accounts_with_charges",
    "is_micro",
    "is_small",
    "is_full_accounts",
    "has_changed_name",
    "num_mortgages_outstanding",
    "num_mortgages_total",
    "has_active_mortgages",
    "fast_growth_proxy",
    "Mortgages.NumMortCharges",
    "Mortgages.NumMortSatisfied",
    "Mortgages.NumMortPartSatisfied",
    "accounts_ever_late",
    "conf_stmt_overdue_days",
    "total_overdue_days",
    "log_overdue"
]

# Keep only columns that actually exist
GROWTH_FEATURES = [
    column for column in GROWTH_FEATURES
    if column in df.columns
]

X = df[GROWTH_FEATURES].copy()
y = df["label_growth_opportunity"].copy()

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("✅ Leakage-safe growth data prepared")
print("Features used:", len(GROWTH_FEATURES))
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nFeatures:")
print(GROWTH_FEATURES)

✅ Leakage-safe growth data prepared
Features used: 32
Training rows: 2516347
Testing rows: 629087

Features:
['has_mortgage_history', 'mortgage_completion_rate', 'mortgage_active_ratio', 'mortgage_per_year', 'company_size_score', 'age_vs_sector_avg', 'is_severely_overdue', 'overdue_intensity', 'sector_avg_age', 'sector_mortgage_rate', 'sector_overdue_rate', 'sector_dormant_rate', 'sic_division', 'active_with_mortgages', 'small_and_growing', 'overdue_and_active', 'full_accounts_with_charges', 'is_micro', 'is_small', 'is_full_accounts', 'has_changed_name', 'num_mortgages_outstanding', 'num_mortgages_total', 'has_active_mortgages', 'fast_growth_proxy', 'Mortgages.NumMortCharges', 'Mortgages.NumMortSatisfied', 'Mortgages.NumMortPartSatisfied', 'accounts_ever_late', 'conf_stmt_overdue_days', 'total_overdue_days', 'log_overdue']


In [8]:

# CHECK FEATURES FOR POSSIBLE DATA LEAKAGE

leakage_check = []

for column in X.columns:
    unique_values = X[column].nunique(dropna=False)

    # Check whether a feature is exactly the same as the target
    exact_match = X[column].equals(y)

    # Calculate correlation for numeric columns
    correlation = X[column].corr(y)

    leakage_check.append({
        "Feature": column,
        "Unique Values": unique_values,
        "Exact Target Match": exact_match,
        "Correlation with Target": correlation
    })

leakage_report = pd.DataFrame(leakage_check)

leakage_report = leakage_report.sort_values(
    "Correlation with Target",
    key=lambda values: values.abs(),
    ascending=False
)

print("Top features most strongly connected to the growth label:")
display(leakage_report.head(15))

Top features most strongly connected to the growth label:


,Feature,Unique Values,Exact Target Match,Correlation with Target
5,age_vs_sector_avg,109434,False,-0.552763
28,accounts_ever_late,2,False,-0.293753
31,log_overdue,3103,False,-0.277888
7,overdue_intensity,165101,False,-0.224413
6,is_severely_overdue,2,False,-0.211665
1,mortgage_completion_rate,1625,False,-0.211630
29,conf_stmt_overdue_days,3488,False,-0.198631
30,total_overdue_days,9062,False,-0.181121
0,has_mortgage_history,2,False,-0.141164
15,overdue_and_active,2,False,-0.134402


In [9]:

# EXTRA TREES MODEL - LEAKAGE-SAFE FEATURES

extra_model = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

print("Training Extra Trees with leakage-safe features...")

extra_model.fit(X_train, y_train)

extra_pred = extra_model.predict(X_test)
extra_prob = extra_model.predict_proba(X_test)[:, 1]

extra_results = {
    "Model": "Extra Trees",
    "Accuracy": accuracy_score(y_test, extra_pred),
    "Precision": precision_score(y_test, extra_pred),
    "Recall": recall_score(y_test, extra_pred),
    "F1 Score": f1_score(y_test, extra_pred),
    "ROC AUC": roc_auc_score(y_test, extra_prob)
}

print("\n✅ Extra Trees Results")
for metric, value in extra_results.items():
    if metric == "Model":
        print(metric, ":", value)
    else:
        print(metric, ":", round(value, 4))

Training Extra Trees with leakage-safe features...

✅ Extra Trees Results
Model : Extra Trees
Accuracy : 0.9129
Precision : 0.9188
Recall : 0.9283
F1 Score : 0.9235
ROC AUC : 0.9603


In [10]:
# LOGISTIC REGRESSION

from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)

print("Training Logistic Regression...")

log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)
log_prob = log_model.predict_proba(X_test)[:,1]

print("\n===== Logistic Regression =====")
print("Accuracy :", round(accuracy_score(y_test, log_pred),4))
print("Precision:", round(precision_score(y_test, log_pred),4))
print("Recall   :", round(recall_score(y_test, log_pred),4))
print("F1 Score :", round(f1_score(y_test, log_pred),4))
print("ROC AUC  :", round(roc_auc_score(y_test, log_prob),4))

Training Logistic Regression...

===== Logistic Regression =====
Accuracy : 0.8849
Precision: 0.8911
Recall   : 0.9079
F1 Score : 0.8994
ROC AUC  : 0.951


In [11]:
# ADABOOST 

from sklearn.ensemble import AdaBoostClassifier

ada_model = AdaBoostClassifier(
    n_estimators=100,
    random_state=42
)

print("Training AdaBoost...")

ada_model.fit(X_train, y_train)

ada_pred = ada_model.predict(X_test)
ada_prob = ada_model.predict_proba(X_test)[:, 1]

print("\n===== AdaBoost =====")
print("Accuracy :", round(accuracy_score(y_test, ada_pred),4))
print("Precision:", round(precision_score(y_test, ada_pred),4))
print("Recall   :", round(recall_score(y_test, ada_pred),4))
print("F1 Score :", round(f1_score(y_test, ada_pred),4))
print("ROC AUC  :", round(roc_auc_score(y_test, ada_prob),4))

Training AdaBoost...

===== AdaBoost =====
Accuracy : 0.9012
Precision: 0.8704
Recall   : 0.9702
F1 Score : 0.9176
ROC AUC  : 0.9617


In [12]:
# COMPARISON 

import pandas as pd

comparison = pd.DataFrame([
    {
        "Model": "Extra Trees",
        "Accuracy": 0.9129,
        "Precision": 0.9188,
        "Recall": 0.9283,
        "F1 Score": 0.9235,
        "ROC AUC": 0.9603
    },
    {
        "Model": "Logistic Regression",
        "Accuracy": 0.8849,
        "Precision": 0.8911,
        "Recall": 0.9079,
        "F1 Score": 0.8994,
        "ROC AUC": 0.9510
    },
    {
        "Model": "AdaBoost",
        "Accuracy": 0.9012,
        "Precision": 0.8704,
        "Recall": 0.9702,
        "F1 Score": 0.9176,
        "ROC AUC": 0.9617
    }
])

comparison = comparison.sort_values("ROC AUC", ascending=False)

display(comparison)

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
2,AdaBoost,0.9012,0.8704,0.9702,0.9176,0.9617
0,Extra Trees,0.9129,0.9188,0.9283,0.9235,0.9603
1,Logistic Regression,0.8849,0.8911,0.9079,0.8994,0.9510


Comparison of the Models

There were three machine learning models considered using the leakage-free features.

| Model         | ROC-AUC     | F1 Score   |
| ------------- | ----------  | ---------- |
| Extra Trees   | 0.9603      | 0.9235     |
| AdaBoost      | 0.9617      | 0.9176     |
| Logistic Regr.| 0.9510      | 0.8994     |

Conclusion

- The best ROC-AUC was obtained by AdaBoost model (0.9617).
- The best F1 Score was reached by Extra Trees (0.9235), i.e., the highest performance in terms of the balance between precision and recall.
- Logistic regression is used as simple benchmark but showed poor performance.

Thus, the ensemble models outperformed Logistic Regression; in particular, Extra Trees provided the best balanced performance.